# 10. Linear Algebra

Most of this lives in `np.linalg`. The examples use small matrices with known answers so you can check the results by hand.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[4., 1.], [2., 3.]])
B = np.array([[1., 2.], [0., 1.]])

## Element-wise vs matrix multiplication

This is the classic beginner mistake: `*` multiplies **element by element**. For matrix multiplication, use `@` (or `np.matmul` / `np.dot`).

In [2]:
print(A * B)    # element-wise
print(A @ B)    # matrix product

[[4. 2.]
 [0. 3.]]
[[4. 9.]
 [2. 7.]]


In [3]:
# these all give the same matrix product for 2-D arrays
print(np.allclose(A @ B, np.matmul(A, B)), np.allclose(A @ B, A.dot(B)))

True True


`@` also works on **stacks** of matrices: the last two dimensions are the matrices, and everything before is treated as a batch. `np.dot` behaves differently for 3-D+ inputs, so stick with `@`.

In [4]:
rng = np.random.default_rng(0)
batch = rng.random((10, 3, 4))
weights = rng.random((4, 2))
(batch @ weights).shape

(10, 3, 2)

## Vector products

In [5]:
u = np.array([1., 2., 3.])
v = np.array([4., 5., 6.])
print('dot:  ', u @ v)            # same as np.dot(u, v)
print('outer:\n', np.outer(u, v))
print('cross:', np.cross(u, v))

dot:   32.0
outer:
 [[ 4.  5.  6.]
 [ 8. 10. 12.]
 [12. 15. 18.]]
cross: [-3.  6. -3.]


## Norms

In [6]:
x = np.array([3., 4.])
print(np.linalg.norm(x))          # Euclidean length: 5
print(np.linalg.norm(x, ord=1))   # sum of absolute values: 7
print(np.linalg.norm(x, ord=np.inf))
print(np.linalg.norm(A))          # Frobenius norm for a matrix

5.0
7.0
4.0
5.477225575051661


In [7]:
# normalising each row of a matrix to unit length
M = rng.random((3, 2))
M_unit = M / np.linalg.norm(M, axis=1, keepdims=True)
np.linalg.norm(M_unit, axis=1)

array([1., 1., 1.])

## Trace, determinant, rank

Note that `trace` is `np.trace`, not `np.linalg.trace` (there is one now, but `np.trace` is the one you'll see everywhere).

In [8]:
print('trace:', np.trace(A))
print('det:  ', np.linalg.det(A))         # 4*3 - 1*2 = 10
print('rank: ', np.linalg.matrix_rank(A))

singular = np.array([[1., 2.], [2., 4.]])
print('rank of a singular matrix:', np.linalg.matrix_rank(singular))

trace: 7.0
det:   10.000000000000002
rank:  2
rank of a singular matrix: 1


## Inverse and pseudo-inverse

In [9]:
A_inv = np.linalg.inv(A)
print(A_inv)
print(A @ A_inv)    # identity, up to rounding

[[ 0.3 -0.1]
 [-0.2  0.4]]
[[ 1.  0.]
 [-0.  1.]]


Singular matrices don't have an inverse, and `inv` raises an error. The pseudo-inverse `pinv` works for any matrix, including non-square ones.

In [10]:
try:
    np.linalg.inv(singular)
except np.linalg.LinAlgError as e:
    print('LinAlgError:', e)

print(np.linalg.pinv(singular))

LinAlgError: Singular matrix
[[0.04 0.08]
 [0.08 0.16]]


## Solving linear systems

To solve `A x = b`, use `np.linalg.solve`. It's faster and more accurate than computing `inv(A) @ b`.

Example: 4x + y = 9 and 2x + 3y = 13.

In [11]:
b = np.array([9., 13.])
x = np.linalg.solve(A, b)
print(x)
print(np.allclose(A @ x, b))

[1.4 3.4]
True


## Least squares

When there are more equations than unknowns there's usually no exact solution. `lstsq` finds the best fit. Here we fit a line `y = m*x + c` to noisy points:

In [12]:
x = np.linspace(0, 10, 20)
y = 2.5 * x + 1 + rng.normal(0, 1, x.size)

X = np.column_stack([x, np.ones_like(x)])   # design matrix [x, 1]
coef, residuals, rank, sv = np.linalg.lstsq(X, y, rcond=None)
print('slope, intercept:', coef)

slope, intercept: [2.3939 1.7227]


(For simple polynomial fits, `np.polyfit(x, y, deg)` does the same in one line.)

In [13]:
np.polyfit(x, y, 1)

array([2.3939, 1.7227])

## Eigenvalues and eigenvectors

`eig` returns the eigenvalues and a matrix whose **columns** are the eigenvectors.

A real matrix can have complex eigenvalues, so depending on your NumPy version you may get complex results even when the imaginary parts are all zero. Use `.real` if you know they should be real.

In [14]:
w, V = np.linalg.eig(A)
print('eigenvalues:', w.real)
print('eigenvectors (columns):\n', V)

# check A v = lambda v for the first pair
print(np.allclose(A @ V[:, 0], w[0] * V[:, 0]))

eigenvalues: [5. 2.]
eigenvectors (columns):
 [[ 0.7071+0.j -0.4472+0.j]
 [ 0.7071+0.j  0.8944+0.j]]
True


If your matrix is symmetric (covariance matrices, for example), use `eigh`. It's faster, always returns real values, and sorts them in ascending order.

In [15]:
S = np.array([[2., 1.], [1., 2.]])
w, V = np.linalg.eigh(S)
print(w)
print(V)

[1. 3.]
[[-0.7071  0.7071]
 [ 0.7071  0.7071]]


## Matrix decompositions

### QR

In [16]:
M = rng.random((4, 3))
Q, R = np.linalg.qr(M)
print(Q.shape, R.shape)
print(np.allclose(Q @ R, M))
print(np.allclose(Q.T @ Q, np.eye(3)))    # Q has orthonormal columns

(4, 3) (3, 3)
True
True


### SVD

`svd` returns `U`, the singular values `s` (as a 1-D array, largest first), and `Vh`, which is already the **transpose** of V. A common mistake is to transpose it again.

In [17]:
M = rng.random((4, 3))
U, s, Vh = np.linalg.svd(M, full_matrices=False)
print(U.shape, s.shape, Vh.shape)
print(np.allclose(U @ np.diag(s) @ Vh, M))

(4, 3) (3,) (3, 3)
True


Keeping only the largest singular values gives the best low-rank approximation, which is the idea behind PCA and image compression:

In [18]:
k = 1
M_k = U[:, :k] @ np.diag(s[:k]) @ Vh[:k]
print('rank-1 approximation error:', np.linalg.norm(M - M_k).round(4))

rank-1 approximation error: 0.5973


### Cholesky

For symmetric positive-definite matrices:

In [19]:
L = np.linalg.cholesky(S)
print(L)
print(np.allclose(L @ L.T, S))

[[1.4142 0.    ]
 [0.7071 1.2247]]
True


## Matrix powers

In [20]:
# Fibonacci numbers from powers of a 2x2 matrix
F = np.array([[1, 1], [1, 0]])
np.linalg.matrix_power(F, 10)

array([[89, 55],
       [55, 34]])